# 🕳️ Pothole Detection — YOLOv11m Final Optimized
### Changes from previous version:
- `epochs=80` with `patience=20` (no early stopping)
- `yolo11l.pt` — better accuracy than nano
- `imgsz=768` — safe for CPU/low RAM
- `batch=16` — stable for larger model
- `optimizer=AdamW` + `cos_lr=True` — faster convergence
- `warmup_epochs=3` — reduced for 15 epoch run
- `dropout=0.1` — light regularization
- Auto-detects latest exp folder — no more wrong folder errors
- **NEW: Pothole location, coordinates, direction and zone detection**
- **NEW: Direction label (Front-Left, Centre, Front-Right etc.)**
- **NEW: GPS-style grid zone mapping**
- **NEW: Severity estimation from box size**
- **NEW: Summary report after prediction**

## 1. Install & Check Environment

In [11]:
!pip install ultralytics pillow --quiet

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
else:
    print('Running on CPU — 15 epochs should take ~8-9 hours on your AMD Ryzen 5')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00a 0:00:01
PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4


In [12]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/jassukhokher
/kaggle/input/datasets/jassukhokher/samplevideo
/kaggle/input/datasets/jassukhokher/potholedataset
/kaggle/input/datasets/jassukhokher/potholedataset/valid
/kaggle/input/datasets/jassukhokher/potholedataset/valid/valid
/kaggle/input/datasets/jassukhokher/potholedataset/valid/valid/labels
/kaggle/input/datasets/jassukhokher/potholedataset/valid/valid/images
/kaggle/input/datasets/jassukhokher/potholedataset/train
/kaggle/input/datasets/jassukhokher/potholedataset/train/train
/kaggle/input/datasets/jassukhokher/potholedataset/train/train/labels
/kaggle/input/datasets/jassukhokher/potholedataset/train/train/images


## 2. Setup Paths & Verify Folders

In [13]:
from pathlib import Path

# Correct BASE path (IMPORTANT)
BASE_DIR = Path('/kaggle/input/datasets/jassukhokher/potholedataset')

# Correct nested paths
train_images = BASE_DIR / 'train/train/images'
train_labels = BASE_DIR / 'train/train/labels'
valid_images = BASE_DIR / 'valid/valid/images'
valid_labels = BASE_DIR / 'valid/valid/labels'

print('── Folder check ──────────────────────')

for folder in [train_images, train_labels, valid_images, valid_labels]:
    count = len(list(folder.glob('*.*'))) if folder.exists() else 0
    status = '✅' if folder.exists() and count > 0 else '❌'
    print(f'{status}  {folder}  ({count} files)')

print('\n── Annotation coverage ───────────────')

for img_dir, lbl_dir, split in [
    (train_images, train_labels, 'train'),
    (valid_images, valid_labels, 'valid')
]:
    if img_dir.exists() and lbl_dir.exists():
        imgs = {p.stem for p in img_dir.glob('*.*')}
        lbls = {p.stem for p in lbl_dir.glob('*.txt')}
        missing = imgs - lbls

        print(f'{split}: {len(imgs)} images, {len(lbls)} labels', end='')
        print(f'  ⚠️  {len(missing)} missing labels' if missing else '  ✅ all matched')
    else:
        print(f'{split}: ❌ Path issue')

── Folder check ──────────────────────
✅  /kaggle/input/datasets/jassukhokher/potholedataset/train/train/images  (1581 files)
✅  /kaggle/input/datasets/jassukhokher/potholedataset/train/train/labels  (1581 files)
✅  /kaggle/input/datasets/jassukhokher/potholedataset/valid/valid/images  (396 files)
✅  /kaggle/input/datasets/jassukhokher/potholedataset/valid/valid/labels  (396 files)

── Annotation coverage ───────────────
train: 1581 images, 1581 labels  ✅ all matched
valid: 396 images, 396 labels  ✅ all matched


## 3. Create data.yaml

In [14]:
from pathlib import Path

# Save YAML in writable directory
yaml_path = Path('/kaggle/working/data.yaml')

content = f"""
# Pothole Detection - YOLOv11s Final

path: /kaggle/input/datasets/jassukhokher/potholedataset
train: train/train/images
val: valid/valid/images

nc: 1
names: ['pothole']
"""

with open(yaml_path, 'w') as f:
    f.write(content)

print('✅ data.yaml created at:', yaml_path)
print(content)

✅ data.yaml created at: /kaggle/working/data.yaml

# Pothole Detection - YOLOv11s Final

path: /kaggle/input/datasets/jassukhokher/potholedataset
train: train/train/images
val: valid/valid/images

nc: 1
names: ['pothole']



## 4. Train — 80 Epochs, YOLOv11m

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')   # better accuracy

results = model.train(
    data='/kaggle/working/data.yaml',

    # 🔥 CORE
    epochs=80,
    patience=20,
    imgsz=768,
    batch=16,

    # 🔥 OPTIMIZER
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,

    # 🔥 LOSS TUNING (important)
    box=7.5,
    cls=0.5,
    dfl=1.5,

    # 🔥 AUGMENTATION (big recall boost)
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    degrees=15,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,

    # 🔥 REGULARIZATION
    dropout=0.1,
    weight_decay=0.0005,

    # OUTPUT
    project='/kaggle/working/pothole_yolo11',
    name='improved',
    save=True
)

   

print('\n✅ Training complete!')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=15, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=7

In [ ]:
!ls /kaggle/working/pothole_yolo11 -R

## 5. Auto-detect Latest Experiment Folder

In [ ]:
# CHANGE: automatically finds the latest exp folder
# so you never have to manually update the folder name again
import os
from pathlib import Path

BASE_DIR   = Path(os.getcwd())
yolo_root  = BASE_DIR / 'pothole_yolo11'

# find the most recently modified experiment folder
exp_dirs   = sorted(
    [d for d in yolo_root.iterdir() if d.is_dir()],
    key=lambda d: d.stat().st_mtime,
    reverse=True
)

if not exp_dirs:
    raise RuntimeError('No experiment folders found. Did training run?')

EXP_DIR    = exp_dirs[0]
BEST_MODEL = EXP_DIR / 'weights' / 'best.pt'
LAST_MODEL = EXP_DIR / 'weights' / 'last.pt'
MODEL_PATH = BEST_MODEL if BEST_MODEL.exists() else LAST_MODEL

print(f'Experiment folder : {EXP_DIR.name}')
print(f'Model used        : {MODEL_PATH.name}')
print(f'Full path         : {MODEL_PATH}')

## 6. Print Performance Metrics

In [ ]:
from ultralytics import YOLO

model   = YOLO(str(MODEL_PATH))
metrics = model.val(data=str(BASE_DIR / 'data.yaml'), plots=True)

print('\n── Model Performance ──────────────────────')
print(f'mAP@50        : {metrics.box.map50:.4f}')
print(f'mAP@50-95     : {metrics.box.map:.4f}')
print(f'Precision     : {metrics.box.mp:.4f}')
print(f'Recall        : {metrics.box.mr:.4f}')
print('───────────────────────────────────────────')
print('Target: mAP@50 > 0.78 (your previous best)')

## 7. Display Training Plots

In [ ]:
import glob
from IPython.display import Image as IPImage, display
from pathlib import Path

# CHANGE: uses auto-detected EXP_DIR — no hardcoded path
png_files = sorted(glob.glob(str(EXP_DIR / '*.png')))

if png_files:
    for img_path in png_files:
        print(f'\n📊 {Path(img_path).name}')
        display(IPImage(filename=img_path, width=900))
else:
    print(f'No plots in {EXP_DIR}. Check training completed.')

In [ ]:
from pathlib import Path

model_paths = list(Path('/kaggle/working').rglob('best.pt'))

if not model_paths:
    raise FileNotFoundError("❌ No trained model found")

MODEL_PATH = str(model_paths[0])
print("✅ Using model:", MODEL_PATH)

## 8. 🆕 Pothole Location, Direction & Coordinate Analysis

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from IPython.display import Image as IPImage, display
import os

# ── FIXED BASE PATH ─────────────────────────────────────
BASE_DIR = Path('/kaggle/input/datasets/jassukhokher/potholedataset')

# ── helper functions ────────────────────────────────────

def get_direction(cx_norm, cy_norm):
    if   cx_norm < 0.33:  col = 'Left'
    elif cx_norm < 0.66:  col = 'Centre'
    else:                 col = 'Right'

    if   cy_norm < 0.33:  row = 'Top'
    elif cy_norm < 0.66:  row = 'Mid'
    else:                 row = 'Front'

    if row == 'Front' and col == 'Centre': return 'Directly Ahead'
    if row == 'Mid'   and col == 'Centre': return 'Centre'
    return f'{row}-{col}'


def get_severity(w_norm, h_norm):
    area = w_norm * h_norm
    if   area < 0.005: return 'Small'
    elif area < 0.02:  return 'Medium'
    elif area < 0.06:  return 'Large'
    else:              return 'Critical'


def get_urgency(direction, severity):
    immediate = direction in ['Directly Ahead', 'Front-Left', 'Front-Right', 'Front-Centre']
    if immediate and severity in ['Large', 'Critical']: return '🔴 AVOID NOW'
    if immediate: return '🟠 Caution'
    if severity == 'Critical': return '🟠 Caution'
    return '🟡 Noted'


def analyse_potholes(result, img_w, img_h):
    potholes = []
    if result.boxes is None or len(result.boxes) == 0:
        return potholes

    for i, box in enumerate(result.boxes):
        x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
        conf = float(box.conf[0])

        cx_px = (x1 + x2) / 2
        cy_px = (y1 + y2) / 2
        w_px  = x2 - x1
        h_px  = y2 - y1

        cx_n  = cx_px / img_w
        cy_n  = cy_px / img_h
        w_n   = w_px  / img_w
        h_n   = h_px  / img_h

        direction = get_direction(cx_n, cy_n)
        severity  = get_severity(w_n, h_n)
        urgency   = get_urgency(direction, severity)

        potholes.append({
            'id': i + 1,
            'confidence': round(conf * 100, 1),
            'direction': direction,
            'severity': severity,
            'urgency': urgency,
            'centre_px': (round(cx_px), round(cy_px)),
        })

    potholes.sort(key=lambda p: p['centre_px'][1], reverse=True)
    return potholes


def print_report(potholes, img_name, img_w, img_h):
    print(f'\n═══════════════════════════════════════')
    print(f'Pothole Report — {img_name}')
    print(f'Image size: {img_w} x {img_h}')
    print(f'Potholes detected: {len(potholes)}')
    print(f'═══════════════════════════════════════')

    if not potholes:
        print('✅ No potholes detected')
        return

    for p in potholes:
        print(f"\n#{p['id']} | {p['direction']} | {p['severity']} | {p['urgency']} | {p['confidence']}%")


# ── FIXED VALID PATH ────────────────────────────────────
valid_imgs = list((BASE_DIR / 'valid/valid/images').glob('*.jpg')) + \
             list((BASE_DIR / 'valid/valid/images').glob('*.png'))

print(f"✅ Found {len(valid_imgs)} validation images")

# ── LOAD MODEL ──────────────────────────────────────────
# Use detected model path
model = YOLO(MODEL_PATH)

# ── RUN PREDICTIONS ─────────────────────────────────────
if not valid_imgs:
    print('❌ No validation images found!')
else:
    sample_imgs = valid_imgs[:5]

    from PIL import Image as PILImage

    for img_path in sample_imgs:
        pil_img = PILImage.open(img_path)
        img_w, img_h = pil_img.size

        results = model.predict(
            source=str(img_path),
            imgsz=640,
            conf=0.25,
            augment = True,
            iou=0.5,
            save=True,
            verbose=False
        )

        potholes = analyse_potholes(results[0], img_w, img_h)
        print_report(potholes, img_path.name, img_w, img_h)

## 9. Display Prediction Images with Boxes

In [ ]:
from pathlib import Path
import glob
from IPython.display import Image as IPImage, display

BASE_DIR = Path('/kaggle/working')   # ✅ FIXED

pred_results = sorted(glob.glob(str(BASE_DIR / 'runs' / 'detect' / 'predict*' / '*.jpg')))
pred_results += sorted(glob.glob(str(BASE_DIR / 'runs' / 'detect' / 'predict*' / '*.png')))

if pred_results:
    print(f'Showing {len(pred_results)} prediction(s):')
    for p in pred_results:
        display(IPImage(filename=p, width=700))
else:
    print('❌ No prediction images found. Check path.')

In [ ]:
def get_driving_advice(potholes):
    if not potholes:
        return "🟢 GO STRAIGHT — Road clear"

    front = [
        p for p in potholes
        if ('Front' in p['direction'] or 'Directly' in p['direction'])
        and p['severity'] in ['Large', 'Critical']
    ]

    if not front:
        return "🟢 SAFE — Maintain lane"

    left = sum(1 for p in front if 'Left' in p['direction'])
    right = sum(1 for p in front if 'Right' in p['direction'])
    center = sum(1 for p in front if 'Centre' in p['direction'] or 'Directly' in p['direction'])

    if center > 0:
        if left == 0:
            return "⬅️ MOVE LEFT"
        elif right == 0:
            return "➡️ MOVE RIGHT"
        else:
            return "⛔ SLOW DOWN"

    if left > right:
        return "➡️ MOVE RIGHT"

    if right > left:
        return "⬅️ MOVE LEFT"

    return "⚠️ SLOW DOWN"

## 10. 🆕 Batch Analyse All Validation Images — Direction Summary

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image as PILImage
from collections import Counter

BASE_DIR = Path('/kaggle/input/datasets/jassukhokher/potholedataset')

valid_imgs = list((BASE_DIR / 'valid/valid/images').glob('*.jpg')) + \
             list((BASE_DIR / 'valid/valid/images').glob('*.png'))

model = YOLO(str(MODEL_PATH))

all_dirs = []
all_sev = []
total_found = 0

print(f'✅ Found {len(valid_imgs)} validation images\n')

# 🔥 THIS IS THE LOOP YOU MODIFY
for img_path in valid_imgs:
    pil_img = PILImage.open(img_path)
    img_w, img_h = pil_img.size

    results = model.predict(
        source=str(img_path),
        imgsz=640,
        conf=0.35,
        iou=0.5,
        verbose=False,
    )

    potholes = analyse_potholes(results[0], img_w, img_h)
    potholes = [
    p for p in potholes 
    if p['confidence'] > 60 and p['severity'] != 'Small'
]
    total_found += len(potholes)

    # 🔥 ADD THIS PART
    advice = get_driving_advice(potholes)

    print(f"\n📸 Image: {img_path.name}")
    print(f"Potholes detected: {len(potholes)}")
    print(f"🚗 Driving Advice: {advice}")

    for p in potholes:
        all_dirs.append(p['direction'])
        all_sev.append(p['severity'])

# ── SUMMARY ──
print(f'\n═══════════════════════════════════════════')
print(f'  Batch Analysis Summary')
print(f'  Images scanned : {len(valid_imgs)}')
print(f'  Total potholes : {total_found}')
print(f'═══════════════════════════════════════════')

## 11. (Optional) Video Prediction with Direction Report

In [ ]:
import cv2
from ultralytics import YOLO
from pathlib import Path
from collections import deque

# ✅ VIDEO PATH (already confirmed)
video_path = Path('/kaggle/input/datasets/jassukhokher/samplevideo/sample_video.mp4')

# ✅ OUTPUT PATH
output_path = '/kaggle/working/final_output.mp4'

# ✅ LOAD MODEL
model = YOLO(str(MODEL_PATH))

# ✅ OPEN VIDEO
cap = cv2.VideoCapture(str(video_path))

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))

# ✅ VIDEO WRITER
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# ✅ STABILITY BUFFER
advice_history = deque(maxlen=5)

def get_stable_advice(potholes):
    advice = get_driving_advice(potholes)
    advice_history.append(advice)
    return max(set(advice_history), key=advice_history.count)

print("🚀 Processing video...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 🔍 DETECTION
    results = model.predict(frame, imgsz=640, conf=0.25, iou=0.5, verbose=False)

    # 📊 ANALYSIS
    potholes = analyse_potholes(results[0], width, height)

    # 🔥 FILTER NOISE
    potholes = [
        p for p in potholes
        if p['confidence'] > 60 and p['severity'] != 'Small'
    ]

    # 🚗 GET STABLE ADVICE
    advice = get_stable_advice(potholes)

    # 🎨 COLOR BASED ON ACTION
    color = (0, 255, 0)  # default green
    if "LEFT" in advice:
        color = (255, 0, 0)
    elif "RIGHT" in advice:
        color = (0, 0, 255)
    elif "SLOW" in advice:
        color = (0, 255, 255)

    # 📝 DRAW TEXT
    cv2.putText(frame, advice, (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

    # 📦 DRAW BOXES
    annotated = results[0].plot()

    # 💾 SAVE FRAME
    out.write(annotated)

cap.release()
out.release()

print("✅ FINAL VIDEO SAVED AT:", output_path)

## 12. Export Best Model to ONNX

In [ ]:
from ultralytics import YOLO
model = YOLO(str(MODEL_PATH))
model.export(format='onnx', imgsz=640, simplify=True)
print('✅ Exported to ONNX')

## 13. Zip Results

In [ ]:
import shutil
from pathlib import Path
import os

BASE_DIR = Path(os.getcwd())
zip_out  = BASE_DIR / 'final_results_backup'
shutil.make_archive(str(zip_out), 'zip', str(EXP_DIR))
print(f'✅ Zipped to: {zip_out}.zip')